# **CODE for Compositional Generalization in Autoregressive Models via Logit Composition**

# **1. Letter Replacing Experiment**


In [ ]:
import torch
from torch import nn
from transformers import GPT2Config, GPT2LMHeadModel
import random

chars = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ ")
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)

def encode(s):
    return torch.tensor([stoi[c] for c in s], dtype=torch.long)

def decode(t):
    return "".join([itos[int(i)] for i in t])

def make_example(seq_len=16):
    s = "".join(random.choice(chars) for _ in range(seq_len))
    return s

def transform_A_to_K(s):
    return s.replace("A", "K")

def transform_M_to_B(s):
    return s.replace("M", "B")

def identity(s):
    return s

def build_dataset(transform_fn, n=2000, seq_len=16):
    xs, ys = [], []
    for _ in range(n):
        s = make_example(seq_len)
        t = transform_fn(s)
        xs.append(encode(s))
        ys.append(encode(t))
    return torch.stack(xs), torch.stack(ys)

def make_model():
    config = GPT2Config(
        vocab_size=vocab_size,
        n_positions=32,
        n_embd=64,
        n_layer=2,
        n_head=2,
    )
    return GPT2LMHeadModel(config)

def train(model, x, y, epochs=5, lr=3e-4):
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    model.train()
    for ep in range(epochs):
        total = 0
        for i in range(len(x)):
            inp = x[i].unsqueeze(0)
            target = y[i].unsqueeze(0)

            out = model(inp).logits  # (1, T, V)
            loss = loss_fn(out.view(-1, vocab_size), target.view(-1))

            opt.zero_grad()
            loss.backward()
            opt.step()

            total += loss.item()
        print(f"epoch {ep+1} loss {total/len(x):.4f}")

x1, y1 = build_dataset(transform_A_to_K)
x2, y2 = build_dataset(transform_M_to_B)
x3, y3 = build_dataset(identity)

m1 = make_model()
m2 = make_model()
m3 = make_model()

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable = total - trainable

    print(f"Total params:        {total:,}")
    print(f"Trainable params:    {trainable:,}")
    print(f"Non-trainable params:{non_trainable:,}")


count_parameters(m1)
count_parameters(m2)
count_parameters(m3)

train(m1, x1, y1)
train(m2, x2, y2)
train(m3, x3, y3)

def run(model, s):
    model.eval()
    inp = encode(s).unsqueeze(0)
    with torch.no_grad():
        out = model(inp).logits.argmax(-1)
    return decode(out[0])

test = "AMMA TEST"
print("input:", test)
print("A->K:", run(m1, test))
print("M->B:", run(m2, test))
print("identity:", run(m3, test))

# **Compose and Perform Inference**

In [ ]:
import torch
import torch.nn.functional as F

def combined_run(m1, m2, m3, s):
    m1.eval(); m2.eval(); m3.eval()

    inp = encode(s).unsqueeze(0)

    with torch.no_grad():
        logits1 = m1(inp).logits
        logits2 = m2(inp).logits
        logits3 = m3(inp).logits

        logp1 = F.log_softmax(logits1, dim=-1)
        logp2 = F.log_softmax(logits2, dim=-1)
        logp3 = F.log_softmax(logits3, dim=-1)

        combined_logp = logp1 + logp2 - logp3

        out_tokens = combined_logp.argmax(dim=-1)

    return decode(out_tokens[0])


tests = [
    "AMMA",
    "HELLO",
    "MAP",
    "GAMMA",
    "AAA MMM",
    "TEST AM"
]

for t in tests:
    print(t, "->", combined_run(m1, m2, m3, t))

# **2. Experiments with LLMs**

The LLM experiments require access to the Gemma 2 checkpoints on Hugging Face. Set an access token in the runtime environment before executing the model-loading cell, for example `export HF_TOKEN=...` in a shell or `os.environ["HF_TOKEN"] = "..."` in a private local session. Do not hard-code or submit access tokens in this notebook.

## **Installs**

In [ ]:
!pip install lm_eval
!pip install evalplus
!pip install datasets

## **Download the Models**

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN in the environment before loading Gemma 2 / MergeBench models.")

BASE = "google/gemma-2-2b"
MODELS_IDS = {
    "base":      BASE,
    "math_ft":   "MergeBench/gemma-2-2b_math",
    "coding_ft": "MergeBench/gemma-2-2b_coding",
}
MAX_NEW = 512
SAMPLE_MERGED = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)

def load_model(model_id):
    tok = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        token=HF_TOKEN,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    model.eval()
    return tok, model

tok_base,  base_model  = load_model(MODELS_IDS["base"])
tok_math,  math_model  = load_model(MODELS_IDS["math_ft"])
tok_code,  code_model  = load_model(MODELS_IDS["coding_ft"])

assert tok_base.vocab_size == tok_math.vocab_size == tok_code.vocab_size, \
    "Tokenizer vocab sizes must match!"

tok = tok_base
models = {
    "base":      base_model,
    "math_ft":   math_model,
    "coding_ft": code_model,
}
print("Models loaded:", list(models.keys()))

## **Define Required Functions**

In [ ]:
@torch.inference_mode()
def merged_gen(prompt, tok, models):
    enc = tok(prompt, return_tensors="pt")
    model_devices = {
        name: model.model.embed_tokens.weight.device
        for name, model in models.items()
    }

    states = {}
    for name, model in models.items():
        dev = model_devices[name]
        out = model(
            input_ids=enc["input_ids"].to(dev),
            attention_mask=enc["attention_mask"].to(dev),
            use_cache=True,
        )
        states[name] = {
            "pkv":    out.past_key_values,
            "attn":   enc["attention_mask"].to(dev),
            "logits": out.logits[:, -1, :].cpu(),
        }

    generated_ids = []
    eos_id = tok.eos_token_id

    for _ in range(MAX_NEW):
        logp_base = torch.log_softmax(states["base"]["logits"],      dim=-1)
        logp_math = torch.log_softmax(states["math_ft"]["logits"],   dim=-1)
        logp_code = torch.log_softmax(states["coding_ft"]["logits"], dim=-1)

        merged_logp  = logp_math + logp_code - logp_base.clamp(min=-1e4)
        merged_probs = torch.softmax(merged_logp, dim=-1)

        next_token = (
            torch.multinomial(merged_probs, num_samples=1)
            if SAMPLE_MERGED
            else torch.argmax(merged_probs, dim=-1, keepdim=True)
        )
        token_id = next_token.item()
        if eos_id is not None and token_id == eos_id:
            break
        generated_ids.append(token_id)

        for name, model in models.items():
            dev = model_devices[name]
            states[name]["attn"] = torch.cat(
                [states[name]["attn"], torch.ones((1, 1), device=dev, dtype=states[name]["attn"].dtype)],
                dim=1,
            )
            out = model(
                input_ids=next_token.to(dev),
                attention_mask=states[name]["attn"],
                past_key_values=states[name]["pkv"],
                use_cache=True,
            )
            states[name]["pkv"]    = out.past_key_values
            states[name]["logits"] = out.logits[:, -1, :].cpu()

    return tok.decode(generated_ids, skip_special_tokens=True)

import lm_eval
from lm_eval.api.model import LM
from lm_eval.api.registry import register_model

@register_model("merged_model")
class MergedModel(LM):
    def __init__(self, models, tok, **kwargs):
        super().__init__()
        self._models = models
        self._tok    = tok

    def loglikelihood(self, requests):
        results = []
        for req in requests:
            context, continuation = req.args
            full    = context + continuation
            enc_f   = self._tok(full,    return_tensors="pt")
            enc_c   = self._tok(context, return_tensors="pt")
            ctx_len = enc_c["input_ids"].shape[1]

            merged_logp = self._merged_logprobs(enc_f["input_ids"])
            cont_logp   = merged_logp[0, ctx_len - 1:-1, :]
            cont_ids    = enc_f["input_ids"][0, ctx_len:]
            score       = cont_logp[range(len(cont_ids)), cont_ids].sum().item()
            results.append((score, True))
        return results

    def loglikelihood_rolling(self, requests):
        raise NotImplementedError

    def generate_until(self, requests):
        results = []
        for req in requests:
            context = req.args[0]
            until   = req.args[1].get("until", [self._tok.eos_token]) if len(req.args) > 1 else [self._tok.eos_token]
            pred    = merged_gen(context, self._tok, self._models)
            for stop in until:
                if stop and stop in pred:
                    pred = pred[:pred.index(stop)]
            results.append(pred)
        return results

    def _merged_logprobs(self, input_ids):
        log_probs = {}
        for name, model in self._models.items():
            dev = model.model.embed_tokens.weight.device
            with torch.inference_mode():
                out = model(input_ids=input_ids.to(dev))
            log_probs[name] = torch.log_softmax(out.logits.cpu(), dim=-1)
        return log_probs["math_ft"] + log_probs["coding_ft"] - log_probs["base"].clamp(min=-1e4)

## **Evaluate on GSM8k**

In [ ]:
harness_results = lm_eval.simple_evaluate(
    model=MergedModel(models=models, tok=tok),  # models & tok from Cell 2
    tasks=["gsm8k"],
    num_fewshot=8,
    batch_size=1,
)

for task, metrics in harness_results["results"].items():
    print(f"{task}: {metrics}")

r = harness_results["results"]["gsm8k"]
print(f"GSM8K strict:   {r['exact_match,strict-match']:.1%}")
print(f"GSM8K flexible: {r['exact_match,flexible-extract']:.1%}")

## **Evaluate on MATH**

In [ ]:
import re
from datasets import load_dataset


MATH_DATASET = "DigitalLearningGmbH/MATH-lighteval"

def extract_boxed(text):
    """Extract \\boxed{...} answer from model output or reference."""
    match = re.search(r"\\boxed\{([^}]*)\}", text)
    return match.group(1).strip() if match else text.strip()

def normalize(ans):
    """Light normalization: strip spaces, lowercase, remove trailing zeros."""
    ans = ans.replace(" ", "").lower()
    try:
        return str(float(ans))
    except ValueError:
        return ans

def score_math(pred, ref):
    return normalize(extract_boxed(pred)) == normalize(extract_boxed(ref))

try:
    math_ds = load_dataset(MATH_DATASET, "all", split="test")
except Exception:
    from datasets import concatenate_datasets
    subjects = [
        "algebra", "counting_and_probability", "geometry",
        "intermediate_algebra", "number_theory", "prealgebra", "precalculus"
    ]
    math_ds = concatenate_datasets([
        load_dataset(MATH_DATASET, subj, split="test") for subj in subjects
    ])

correct = 0
total   = 0
results_log = []

FEW_SHOT_PREFIX = """Solve the following math problem step by step. Put your final answer in \\boxed{}.
Problem: What is $2^{10}$?
Solution: $2^{10} = 1024$. The answer is $\\boxed{1024}$.
Problem: Simplify $\\frac{x^2 - 1}{x - 1}$.
Solution: $\\frac{x^2-1}{x-1} = \\frac{(x+1)(x-1)}{x-1} = x+1$. The answer is $\\boxed{x+1}$.
"""
for ex in math_ds.select(range(len(math_ds) )):
    prompt = FEW_SHOT_PREFIX + f"Problem: {ex['problem']}\nSolution:"
    pred   = merged_gen(prompt, tok, models)
    ref    = ex["solution"]
    ok     = score_math(pred, ref)
    correct += int(ok)
    total   += 1
    results_log.append({
        "problem":  ex["problem"],
        "level":    ex["level"],
        "type":     ex["type"],
        "pred":     pred,
        "ref":      ref,
        "correct":  ok,
    })

print(f"\nMATH accuracy: {correct}/{total} = {correct/total:.1%}")

import pandas as pd
df = pd.DataFrame(results_log)
print("\nBy subject:")
print(df.groupby("type")["correct"].mean().round(3))
print("\nBy difficulty level:")
print(df.groupby("level")["correct"].mean().round(3))

import json
with open("math_eval_results.json", "w") as f:
    json.dump(results_log, f, indent=2)
print("Saved to math_eval_results.json")

## **Evaluate on HumanEval+**

In [ ]:
import json
from evalplus.data import get_human_eval_plus, get_mbpp_plus
from tqdm import tqdm

for benchmark, getter in [("humaneval", get_human_eval_plus)]:
    print(f"\n Loading {benchmark} dataset...", end=" ", flush=True)
    problems = getter()
    print(f"done ({len(problems)} problems)")

    items = list(problems.items())

    samples = []
    failed  = 0

    pbar = tqdm(items, desc=f"Generating [{benchmark}]", unit="problem",
                dynamic_ncols=True)

    for task_id, problem in pbar:
        try:
            completion = merged_gen(problem["prompt"], tok, models)
            samples.append({"task_id": task_id, "completion": completion})
        except Exception as e:
            failed += 1
            tqdm.write(f" {task_id} failed: {e}")
            samples.append({"task_id": task_id, "completion": ""})

        pbar.set_postfix(done=len(samples), failed=failed)

    out_file = f"samples_{benchmark}.jsonl"
    with open(out_file, "w") as f:
        for s in samples:
            f.write(json.dumps(s) + "\n")

    print(f"\n[{benchmark}] {len(samples)} samples → {out_file}")
    if failed:
        print(f"{failed} problems failed and were saved as empty strings")

    print(f"Run: evalplus.evaluate --dataset {benchmark} --samples {out_file}\n")

In [ ]:
!python -m evalplus.evaluate --dataset humaneval --samples samples_humaneval.jsonl

## **Evaluate on MBPP+**

In [ ]:
import json
from evalplus.data import get_human_eval_plus, get_mbpp_plus
from tqdm import tqdm

for benchmark, getter in [("mbpp", get_mbpp_plus)]:
    print(f"\n Loading {benchmark} dataset...", end=" ", flush=True)
    problems = getter()
    print(f"done ({len(problems)} problems)")

    items = list(problems.items())

    samples = []
    failed  = 0

    pbar = tqdm(items, desc=f"Generating [{benchmark}]", unit="problem",
                dynamic_ncols=True)

    for task_id, problem in pbar:
        try:
            completion = merged_gen(problem["prompt"], tok, models)
            samples.append({"task_id": task_id, "completion": completion})
        except Exception as e:
            failed += 1
            tqdm.write(f" {task_id} failed: {e}")
            samples.append({"task_id": task_id, "completion": ""})

        pbar.set_postfix(done=len(samples), failed=failed)

    out_file = f"samples_{benchmark}.jsonl"
    with open(out_file, "w") as f:
        for s in samples:
            f.write(json.dumps(s) + "\n")

    print(f"\n[{benchmark}] {len(samples)} samples → {out_file}")
    if failed:
        print(f"{failed} problems failed and were saved as empty strings")

    print(f"Run: evalplus.evaluate --dataset {benchmark} --samples {out_file}\n")

In [ ]:
!python -m evalplus.evaluate --dataset mbpp --samples samples_mbpp.jsonl